# Q-Learning: Model-Free Reinforcement Learning

## Overview
Q-Learning is a **model-free, off-policy reinforcement learning algorithm** that learns the value of actions in states without requiring knowledge of the environment model (transition probabilities). The agent learns by interacting with the environment and updating a Q-table based on the rewards it receives.

### Key Concepts:
- **Model-Free**: No need to know the environment's dynamics
- **Off-Policy**: Learns optimal policy while following an exploratory policy
- **Q-Value**: Expected cumulative reward for taking action 'a' in state 's'
- **Exploration vs Exploitation**: Balance between trying new actions (exploration) and using known good actions (exploitation)

This notebook will guide you through implementing Q-learning from scratch and training an agent to solve the FrozenLake environment.

## 1. Import Required Libraries

We'll need the following libraries:
- **NumPy**: For numerical operations and matrix handling
- **Matplotlib**: For visualizing training progress
- **Gym**: OpenAI's environment toolkit (or gymnasium for newer versions)
- **Random**: For epsilon-greedy action selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict

# Try importing from gymnasium (newer version) or gym (older version)
try:
    import gymnasium as gym
except ImportError:
    import gym

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

## 2. Define the Environment

We'll use **FrozenLake-v1**, a classic environment where:
- **Goal**: Navigate from START to GOAL on a frozen lake
- **States**: 16 positions (4x4 grid)
- **Actions**: 4 movements (LEFT, DOWN, RIGHT, UP)
- **Rewards**: +1 for reaching GOAL, 0 otherwise
- **Challenges**: Slippery surface (stochastic environment)

### Environment Layout:
```
S F F F
F H F H
F F F H
H F F G
```
- **S**: Start position
- **F**: Frozen surface (safe)
- **H**: Hole (game over if you step on it)
- **G**: Goal

In [ ]:
# Create the FrozenLake environment
env = gym.make('FrozenLake-v1', is_slippery=False)  # is_slippery=False for deterministic moves

# Get environment info
print(f"Environment: FrozenLake")
print(f"Number of States: {env.observation_space.n}")
print(f"Number of Actions: {env.action_space.n}")
print(f"State Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")

# Action mapping
action_mapping = {0: "LEFT", 1: "DOWN", 2: "RIGHT", 3: "UP"}
print(f"\nAction Mapping: {action_mapping}")

## 3. Initialize Q-Table

The Q-table is a lookup table where:
- **Rows** represent states (0-15)
- **Columns** represent actions (0-3)
- **Values** are Q(s,a) - the expected cumulative reward

### Bellman Equation Foundation:
The Q-learning update rule is based on the Bellman equation:

$$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$$

Where:
- $\alpha$ (alpha) = Learning rate (0 to 1): How much to update Q-values
- $\gamma$ (gamma) = Discount factor (0 to 1): Importance of future rewards
- $r$ = Immediate reward
- $\max_{a'} Q(s',a')$ = Maximum Q-value for next state

In [ ]:
# Initialize Q-table with zeros
n_states = env.observation_space.n
n_actions = env.action_space.n

Q_table = np.zeros((n_states, n_actions))

print(f"Q-Table Shape: {Q_table.shape} (states × actions)")
print(f"\nInitial Q-Table:\n{Q_table}")
print(f"\nAll values start at 0. They will be updated during training.")

## 4. Implement Q-Learning Algorithm

The Q-learning algorithm consists of:

1. **Epsilon-Greedy Strategy**: 
   - With probability $\epsilon$: Explore (random action)
   - With probability $1-\epsilon$: Exploit (best known action)

2. **Q-Value Update**:
   - Calculate temporal difference error
   - Update Q(s,a) using the Bellman equation

3. **Episode Loop**:
   - Agent starts in initial state
   - Repeat until episode ends (reaches goal or falls in hole)
   - Update Q-table after each step

In [ ]:
# Q-Learning Parameters
learning_rate = 0.8      # alpha: how much to update Q-values (0 to 1)
discount_factor = 0.95   # gamma: importance of future rewards (0 to 1)
epsilon = 1.0            # exploration rate (1.0 = fully explore at start)
epsilon_decay = 0.995    # decay epsilon after each episode
epsilon_min = 0.01       # minimum epsilon (don't stop exploring completely)

def epsilon_greedy_action(state, Q_table, epsilon):
    """
    Epsilon-greedy action selection strategy.
    
    Args:
        state: Current state
        Q_table: Q-value table
        epsilon: Exploration probability
    
    Returns:
        action: Selected action (0-3)
    """
    if random.random() < epsilon:
        # Explore: random action
        return env.action_space.sample()
    else:
        # Exploit: best known action
        return np.argmax(Q_table[state])

def q_learning_step(state, action, reward, next_state, done, Q_table, learning_rate, discount_factor):
    """
    Perform a single Q-learning update step.
    
    Q(s,a) ← Q(s,a) + α[r + γ·max(Q(s',a')) - Q(s,a)]
    
    Args:
        state: Current state
        action: Action taken
        reward: Reward received
        next_state: Next state
        done: Whether episode ended
        Q_table: Q-value table
        learning_rate: Learning rate (alpha)
        discount_factor: Discount factor (gamma)
    
    Returns:
        Updated Q_table
    """
    # Calculate the TD (Temporal Difference) target
    if done:
        # If episode ended, no future rewards
        td_target = reward
    else:
        # TD target = immediate reward + discounted max future Q-value
        td_target = reward + discount_factor * np.max(Q_table[next_state])
    
    # Calculate TD error
    td_error = td_target - Q_table[state, action]
    
    # Update Q-value
    Q_table[state, action] += learning_rate * td_error
    
    return Q_table

print("Q-Learning functions defined!")
print(f"\nHyperparameters:")
print(f"  Learning Rate (α): {learning_rate}")
print(f"  Discount Factor (γ): {discount_factor}")
print(f"  Initial Epsilon (ε): {epsilon}")
print(f"  Epsilon Decay: {epsilon_decay}")

## 5. Train the Agent

We'll run multiple episodes where the agent:
1. Starts at the initial state
2. Selects actions using epsilon-greedy strategy
3. Receives rewards from the environment
4. Updates Q-values using the Q-learning rule
5. Gradually increases exploitation (decreases epsilon)

**Training Metrics**:
- Episode reward: 1 if goal reached, 0 otherwise
- Success rate: Percentage of episodes where goal was reached

In [ ]:
# Training parameters
num_episodes = 1000
max_steps_per_episode = 100

# Lists to track training progress
episode_rewards = []
episode_lengths = []
success_count = 0

# Reset environment and epsilon
env.reset()
epsilon = 1.0

print("Starting Q-Learning Training...")
print(f"Total episodes: {num_episodes}\n")

# Training loop
for episode in range(num_episodes):
    # Reset environment for new episode
    state, info = env.reset()
    episode_reward = 0
    episode_steps = 0
    
    # Run episode until termination or max steps
    for step in range(max_steps_per_episode):
        # Select action using epsilon-greedy strategy
        action = epsilon_greedy_action(state, Q_table, epsilon)
        
        # Take action in environment
        try:
            # For gymnasium
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        except:
            # For older gym versions
            next_state, reward, done, info = env.step(action)
        
        # Update Q-table
        Q_table = q_learning_step(state, action, reward, next_state, done, 
                                  Q_table, learning_rate, discount_factor)
        
        episode_reward += reward
        episode_steps += 1
        state = next_state
        
        if done:
            break
    
    # Track metrics
    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_steps)
    if episode_reward > 0:
        success_count += 1
    
    # Decay epsilon (reduce exploration over time)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    # Print progress every 100 episodes
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(episode_rewards[-100:])
        success_rate = np.sum(episode_rewards[-100:]) / 100
        print(f"Episode {episode + 1:4d} | Avg Reward: {avg_reward:.3f} | Success Rate: {success_rate:.2%} | Epsilon: {epsilon:.4f}")

print(f"\nTraining Complete!")
print(f"Total Successes: {success_count}/{num_episodes} ({success_count/num_episodes:.2%})")

In [ ]:
# Examine learned Q-table
print("Learned Q-Table (sample states):")
print("\nState | LEFT  | DOWN  | RIGHT | UP")
print("------|-------|-------|-------|-------")
for state in range(16):
    q_values = Q_table[state]
    print(f"  {state:2d}  | {q_values[0]:5.2f} | {q_values[1]:5.2f} | {q_values[2]:5.2f} | {q_values[3]:5.2f}")

# Show policy (best action for each state)
print("\n\nLearned Policy (best action for each state):")
policy = np.argmax(Q_table, axis=1)
action_names = ['L', 'D', 'R', 'U']  # LEFT, DOWN, RIGHT, UP
print("\nGrid representation (4x4):")
for i in range(4):
    row = ""
    for j in range(4):
        state_idx = i * 4 + j
        action = policy[state_idx]
        row += f"{action_names[action]} "
    print(row)

## 6. Evaluate the Trained Agent

Now we'll test the trained agent by running episodes with **greedy action selection** (no exploration). We'll measure:
- Success rate: Percentage of episodes where the agent reached the goal
- Average steps to goal: How efficiently the agent navigates
- Consistency: How reliably the agent performs

In [ ]:
# Evaluation phase: test with learned policy (no exploration)
num_test_episodes = 100
test_rewards = []
test_lengths = []

print("Evaluating Trained Agent (100 test episodes)...")

for episode in range(num_test_episodes):
    state, info = env.reset()
    episode_reward = 0
    episode_steps = 0
    
    for step in range(max_steps_per_episode):
        # Use greedy policy (best action only, no exploration)
        action = np.argmax(Q_table[state])
        
        try:
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        except:
            next_state, reward, done, info = env.step(action)
        
        episode_reward += reward
        episode_steps += 1
        state = next_state
        
        if done:
            break
    
    test_rewards.append(episode_reward)
    test_lengths.append(episode_steps)

# Calculate evaluation metrics
success_rate = np.sum(test_rewards) / num_test_episodes
avg_steps_to_goal = np.mean([length for reward, length in zip(test_rewards, test_lengths) if reward > 0])
avg_steps_all = np.mean(test_lengths)

print(f"\n=== EVALUATION RESULTS ===")
print(f"Success Rate: {success_rate:.2%} ({int(np.sum(test_rewards))}/{num_test_episodes} episodes)")
print(f"Average Steps (on successful episodes): {avg_steps_to_goal:.1f}")
print(f"Average Steps (all episodes): {avg_steps_all:.1f}")
print(f"Min Steps to Goal: {min([length for reward, length in zip(test_rewards, test_lengths) if reward > 0])}")
print(f"Max Steps to Goal: {max([length for reward, length in zip(test_rewards, test_lengths) if reward > 0])}")

## 7. Visualize Results

We'll create visualizations to understand the training progress and agent performance:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Episode Rewards over training (with moving average)
ax1 = axes[0, 0]
ax1.plot(episode_rewards, alpha=0.3, label='Episode Reward')
# Calculate and plot moving average
window = 50
moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax1.plot(range(window-1, num_episodes), moving_avg, label=f'Moving Avg (window={window})', linewidth=2, color='orange')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward')
ax1.set_title('Training Progress: Episode Rewards')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Success Rate (rolling window)
ax2 = axes[0, 1]
window = 50
success_rate_rolling = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax2.plot(range(window-1, num_episodes), success_rate_rolling * 100, linewidth=2, color='green')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Success Rate (%)')
ax2.set_title('Training Progress: Success Rate (50-episode window)')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 105])

# Plot 3: Episode Length Distribution
ax3 = axes[1, 0]
ax3.hist(episode_lengths, bins=30, alpha=0.7, edgecolor='black')
ax3.set_xlabel('Steps per Episode')
ax3.set_ylabel('Frequency')
ax3.set_title('Distribution of Episode Lengths During Training')
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Q-value Distribution
ax4 = axes[1, 1]
q_values_flat = Q_table.flatten()
ax4.hist(q_values_flat[q_values_flat > 0], bins=30, alpha=0.7, color='purple', edgecolor='black')
ax4.set_xlabel('Q-Value')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Non-Zero Q-Values in Learned Table')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Visualization complete!")

## Summary & Key Takeaways

### What is Q-Learning?
Q-Learning is a **model-free, off-policy reinforcement learning algorithm** that learns the optimal policy by estimating Q-values (state-action values) without needing to know the environment's dynamics.

### Algorithm Steps:
1. **Initialize** Q-table with zeros
2. **For each episode**:
   - Select actions using ε-greedy strategy (explore vs exploit)
   - Observe reward and next state
   - Update Q-value using: $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$
   - Gradually reduce ε (less exploration over time)

### Key Hyperparameters:
- **Learning Rate (α)**: How much to update Q-values (typically 0.1-1.0)
- **Discount Factor (γ)**: Value of future rewards (typically 0.9-0.99)
- **Epsilon (ε)**: Exploration rate (start high, decay over time)

### Advantages:
✓ Model-free (no need for environment model)  
✓ Off-policy (can learn from any behavior policy)  
✓ Converges to optimal policy (under right conditions)  
✓ Works well for discrete action/state spaces  

### Limitations:
✗ Not suitable for continuous action spaces (use Policy Gradient methods)  
✗ Requires discretized state space  
✗ Can be slow to converge with large state spaces  
✗ May suffer from overestimation of Q-values (addressed by Double Q-Learning)  

### Next Steps:
Try experimenting with:
- Different learning rates and discount factors
- Various epsilon decay schedules
- Different environments (CartPole, MountainCar, etc.)
- Double Q-Learning to reduce overestimation
- Experience Replay for better sample efficiency